# Call a confidential analysis session over MCP

Run this notebook **on-premises or on a VPN-connected workstation**. Python calculations run in a dedicated, attested **Azure confidential ACI group (AMD SEV-SNP)**, not in this kernel. The gateway/controller and returned results are outside that confidential boundary.

Use synthetic data only. `sample_sales.xlsx` has four invented region/revenue/unit rows; revenue totals 640 and units total 64.

## Before starting

From the repository root, create a Python 3.11+ environment, install `examples\requirements-notebooks.txt`, and select that environment as this notebook's kernel. See [getting started](../docs/getting-started.md) for exact commands. Copy `.env.example` to `.env` in `examples`, then set the private gateway URL, verified PEM, sandbox API ID, and MCP app credentials. The app needs `Sandbox.Access`. The private gateway must resolve and be reachable from this machine.

**Authentication:** this code obtains an Entra token for `api://<MCP_API_APP_ID>/.default` and sends it as an HTTPS bearer token. VPN access is not API authorization. An MCP app secret is kept locally, never sent as a model argument. This notebook makes no model calls.

Running the session cell can create billable workers. Review the [pilot guide](../docs/pilot-guide.md) before using a deployed service.

In [ ]:
import json
import secrets
import sys
from pathlib import Path

root = Path.cwd().resolve()
if root.name == 'examples':
    root = root.parent
if not (root / 'examples' / 'pool_client.py').is_file():
    raise RuntimeError('Start the kernel in the repository root or examples directory')
sys.path.insert(0, str(root / 'examples'))

from pool_client import EXAMPLES, PoolClient, PoolConfig
from pooled_analysis import ANALYSIS
from agent_framework_analysis import validate_summary

config = PoolConfig.from_env()
fetch_attestation = False  # Optional evidence retrieval; this is NOT client verification.
print('Configuration loaded without displaying credentials.')

## Discover tools, allocate, run, download, release

One protected block contains every step that needs a lease. `pool.session()` calls `create_session`, polls `get_session_status` until active, and calls `delete_session` in `finally`, including after a failed operation. A warm session historically took about one minute; cold replacement took about four minutes. These are pilot observations, not a latency guarantee.

Expected result: CSV rows `count`, `mean`, `sum`; revenue values `4,160,640`; units values `4,16,64`. The downloaded artifact is checked against these known synthetic values. Releases schedule asynchronous group destruction; they do not prove Azure deletion has completed.

In [ ]:
async with PoolClient(config) as pool:
    print('Available tools:', ', '.join(await pool.list_tools()))
    async with pool.session() as session_id:
        status = json.loads(await pool.call('get_session_status', session_id=session_id))
        print('Session status:', status['status'])
        await pool.upload(session_id, EXAMPLES / 'sample_sales.xlsx')
        await pool.call('write_file', session_id=session_id, path='notes.txt', content='Synthetic demonstration only.')
        print(await pool.call('read_file', session_id=session_id, path='notes.txt'))
        print(await pool.call('list_dir', session_id=session_id, path='.'))
        print(await pool.call('run_python', session_id=session_id, code=ANALYSIS, timeout=120))
        if fetch_attestation:
            evidence = await pool.call('get_attestation', session_id=session_id, nonce=secrets.token_hex(32))
            print('Evidence retrieved; not verified by this notebook. Bytes:', len(evidence.encode()))
        artifact = await pool.download(session_id, 'summary.csv', EXAMPLES / 'summary.csv')
        print(validate_summary(artifact))
print('Artifact verified; session released.')

## What the other tools do

`run_command` and `run_tests` execute inside the same worker. `pip_install` and `git_clone` are exposed but fail under the no-package/no-Git-egress design; dependencies must be baked into the approved image. `upload_file` and `download_file` transfer base64; the helper handles encoding. `list_sessions` shows sessions owned by your authenticated principal.

**Attestation:** fetching or decoding a token does not verify it. Before admission, the controller verifies fresh MAA evidence bound to the worker TLS key, approved CCE policy, and minimum security versions. An independently verifying client needs that trusted baseline and signature/challenge/key checks; this notebook does not implement an independent verifier.

## Failures and interruption

A 401 usually means a token/tenant/audience issue; a 403 may mean missing permission. Check `Sandbox.Access`, API admin consent, private DNS/VPN, and the administrator-published TLS fingerprint. Never disable certificate verification. A full pool queues requests; expired/failed sessions raise rather than silently retrying code.

Normal exceptions and cancellation release the lease. If the kernel is killed or connectivity fails during creation/release, cleanup cannot be guaranteed. Use a new `PoolClient` block to call `list_sessions`, then explicitly `delete_session` for your unused leases. Clients sharing one app identity share ownership: do not delete another application user's work. The configured idle/absolute TTL is the service-side backstop. An operator should confirm physical worker deletion.

Committed notebooks intentionally have no execution output. Offline checks are not proof of a successful live confidential session; see the [live release gate](../docs/pilot-guide.md#live-release-gate).